In [1]:
# [Cell 1]
# Databricks Free Edition: delta-rs runs locally, Spark runs on serverless via Databricks Connect (see dldg.py)
# delta-rs can't reach UC volumes (the metastore has external access disabled), so tables are copied
# to and from the volume with upload_dir() / download_dir()
from dldg import spark, WORK, tree, rm, upload_dir, download_dir
CH = f"{WORK}/ch06"  # volume folder for this chapter
GEN = "./data/gen"   # local folder for tables written by delta-rs (git-ignored)

In [2]:
# [Cell 2]
# run first for imports
import shutil
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake


def fresh(path):
    """Delete a local table so the book's default mode='error' writes can be re-run."""
    shutil.rmtree(path, ignore_errors=True)

*[Cell 3]*

# Chapter 6: Building Native Applications with Delta Lake
> These exercises use the Python `deltalake` package (delta-rs): no Spark, no JVM. Everything up to
> "Delta Lake on Databricks" runs on your machine. The rest moves a delta-rs table to Databricks, changes it
> with Spark, and reads Spark's commit back with delta-rs.

The book uses `deltalake` 0.x; this notebook uses 1.5.1. `DeltaTable.files()` is gone (use `file_uris()` or
`get_add_actions()`), and `write_deltalake()` dropped `max_rows_per_file` and `schema`.

The CO2 data comes from NOAA. Download it once from the `ch06` folder with `make download`, or:
`curl -o data/co2_mm_mlo.csv https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv`

*[Cell 4]*

## Getting Started
`data/deltatbl-partitioned` is a small table partitioned by `c2`.

In [3]:
# [Cell 5]
dt = DeltaTable("./data/deltatbl-partitioned")
# book: dt.files()
dt.get_add_actions(flatten=True).column("path").to_pylist()

['c2=foo0/part-00000-2bcc9ff6-0551-4401-bd22-d361a60627e3.c000.snappy.parquet',
 'c2=foo1/part-00000-786c7455-9587-454f-9a4c-de0b22b62bbd.c000.snappy.parquet',
 'c2=foo0/part-00001-ca647ee7-f1ad-4d70-bf02-5d1872324d6f.c000.snappy.parquet',
 'c2=foo1/part-00001-1c702e73-89b5-465a-9c6a-25f7559cd150.c000.snappy.parquet']

In [4]:
# [Cell 6]
df = dt.to_pandas()
df

,c1,c2
0,0,foo0
1,2,foo0
2,4,foo0
3,1,foo1
4,3,foo1
5,6,foo0
6,8,foo0
7,5,foo1
8,7,foo1
9,9,foo1


*[Cell 7]*

### Reading large data sets
Partition filters skip whole directories, `columns` reads only some columns, and `filters` uses the file
statistics to skip files and then filters rows.

In [5]:
# [Cell 8]
dt.to_pandas(partitions=[("c2", "=", "foo0")])

,c1,c2
0,0,foo0
1,2,foo0
2,4,foo0
3,6,foo0
4,8,foo0


In [6]:
# [Cell 9]
# book: dt.files([("c2", "=", "foo0")])
dt.file_uris([("c2", "=", "foo0")])

['/home/shridhar/PycharmProjects/delta-lake-definitive-guide/ch06/data/deltatbl-partitioned/c2=foo0/part-00000-2bcc9ff6-0551-4401-bd22-d361a60627e3.c000.snappy.parquet',
 '/home/shridhar/PycharmProjects/delta-lake-definitive-guide/ch06/data/deltatbl-partitioned/c2=foo0/part-00001-ca647ee7-f1ad-4d70-bf02-5d1872324d6f.c000.snappy.parquet']

In [7]:
# [Cell 10]
dt.to_pandas(partitions=[("c2", "=", "foo0")], columns=["c1"])

,c1
0,0
1,2
2,4
3,6
4,8


In [8]:
# [Cell 11]
dt.to_pandas(partitions=[("c2", "=", "foo0")], columns=["c1"], filters=[("c1", "<=", 4), ("c1", ">", 0)])

,c1
0,2
1,4


*[Cell 12]*

#### Generating a table with interesting file statistics
The book writes 4 rows per file with `max_rows_per_file=4`. `deltalake` 1.x writes at least one file per
partition per write, so the months are appended in three batches of four instead: each `year` partition gets
up to three 4-row files.

In [9]:
# [Cell 13]
df = pd.read_csv("data/co2_mm_mlo.csv", comment="#")
fresh(f"{GEN}/filestats")
for months in ([1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]):
    write_deltalake(f"{GEN}/filestats", data=df[df.month.isin(months)].reset_index(drop=True), partition_by=["year"], mode="append")

In [10]:
# [Cell 14]
dt = DeltaTable(f"{GEN}/filestats")
print(len(dt.file_uris()), "files")  # the book had 198; the NOAA file grows every month
dt.to_pandas(filters=[("year", "=", 2022), ("month", ">=", 9)])

207 files


,year,month,decimal date,average,deseasonalized,ndays,sdev,unc
0,2022,9,2022.7083,415.91,419.30,28,0.41,0.15
1,2022,10,2022.7917,415.74,418.97,30,0.27,0.10
2,2022,11,2022.8750,417.47,419.51,25,0.52,0.20
3,2022,12,2022.9583,418.99,419.79,24,0.57,0.22


In [11]:
# [Cell 15]
# The min/max month stats in each 2022 file are what let the filter above read only one of the three files
stats = pa.table(dt.get_add_actions(flatten=True)).to_pandas()
stats[lambda s: s["partition.year"] == 2022][["path", "num_records", "min.month", "max.month"]]

,path,num_records,min.month,max.month
25,year=2022/part-00000-ab3e9b4b-44eb-4940-9922-7...,4,9,12
108,year=2022/part-00000-b5055fa4-7cf2-4baa-9422-0...,4,5,8
160,year=2022/part-00000-409bce4c-3294-4ad3-a8cf-9...,4,1,4


*[Cell 16]*

### Writing data

In [12]:
# [Cell 17]
df = pd.read_csv("./data/co2_mm_mlo.csv", comment="#")
fresh(f"{GEN}/co2_monthly")
write_deltalake(f"{GEN}/co2_monthly", df)
dt = DeltaTable(f"{GEN}/co2_monthly")
print(len(dt.file_uris()), "file(s),", len(dt.to_pandas()), "rows")

1 file(s), 823 rows


*[Cell 18]*

With partitioning

In [14]:
# [Cell 19]
fresh(f"{GEN}/co2_monthly_partitioned")
write_deltalake(f"{GEN}/co2_monthly_partitioned", data=df, partition_by=["year"])

*[Cell 20]*

### Going beyond Pandas
`mode="overwrite"` replaces the data in a new version; the old files stay for time travel until a vacuum.

In [15]:
# [Cell 21]
write_deltalake(f"{GEN}/co2_monthly_partitioned", data=df, mode="overwrite", partition_by=["year"])
dt = DeltaTable(f"{GEN}/co2_monthly_partitioned")
pd.DataFrame(dt.history())[["version", "operation", "clientVersion"]]

,version,operation,clientVersion
0,1,WRITE,delta-rs.py-1.5.1
1,0,WRITE,delta-rs.py-1.5.1


*[Cell 22]*

## Delta Lake on Databricks
A Delta table is just files, so the table delta-rs wrote can be copied to a volume and used by Spark as is.

In [16]:
# [Cell 23]
CO2 = f"{CH}/co2_monthly_partitioned"
rm(CO2)
upload_dir(f"{GEN}/co2_monthly_partitioned", CO2)
tree(f"{CO2}/_delta_log")

00000000000000000000.json
00000000000000000001.json


In [17]:
# [Cell 24]
spark.sql(f"DESCRIBE DETAIL delta.`{CO2}`").select("format", "numFiles", "partitionColumns", "minReaderVersion", "minWriterVersion").show()

+------+--------+----------------+----------------+----------------+
|format|numFiles|partitionColumns|minReaderVersion|minWriterVersion|
+------+--------+----------------+----------------+----------------+
| delta|      69|          [year]|               1|               2|
+------+--------+----------------+----------------+----------------+



In [18]:
# [Cell 25]
spark.sql(f"""
    SELECT year, round(avg(average), 2) AS avg_co2_ppm
    FROM delta.`{CO2}`
    WHERE year >= 2020
    GROUP BY year
    ORDER BY year
""").show()

+----+-----------+
|year|avg_co2_ppm|
+----+-----------+
|2020|     414.21|
|2021|     416.41|
|2022|     418.53|
|2023|     421.08|
|2024|      424.6|
|2025|     427.35|
|2026|     429.53|
+----+-----------+



In [ ]:
# [Cell 26]
# Spark appends a copy of 2025's rows, adding a version written by Databricks Runtime
spark.read.format("delta").load(CO2).where("year = 2025").write.format("delta").mode("append").save(CO2)
spark.sql(f"DESCRIBE HISTORY delta.`{CO2}`").select("version", "operation", "engineInfo").show(truncate=False)

*[Cell 27]*

### Back to delta-rs
Copy the table back and read it with delta-rs: it sees Spark's commit as the latest version.

In [ ]:
# [Cell 28]
download_dir(CO2, f"{GEN}/co2_from_databricks")
dt = DeltaTable(f"{GEN}/co2_from_databricks")
print("version", dt.version())
pd.DataFrame(dt.history())[["version", "operation", "engineInfo", "clientVersion"]]

In [ ]:
# [Cell 29]
# 2 rows per month after Spark's append
dt.to_pandas(filters=[("year", "=", 2025)]).groupby("month").size()

In [ ]:
# [Cell 30]
# clean up the chapter's volume folder and local tables
# rm(CH)
# import os; shutil.rmtree(GEN); os.makedirs(GEN); open(f"{GEN}/.gitkeep", "w").close()